# 01 — Data Inventory
## NFL Big Data Bowl 2027

**Purpose**: Inspect every competition file — schema, row counts, memory usage, missing values, categorical distributions, and join integrity across tracking and player outcome datasets.

**Output**: `outputs/reports/data_inventory.md` and missing data heatmaps in `outputs/figures/`.

> **Environment**: Fully configured for both local execution and direct Kaggle notebook execution with automated `/kaggle/input/` detection.

In [ ]:
# ── Environment & Path Discovery ──────────────────────────────────────────────
import sys
import os
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root and current working directory are on Python path
for p in [Path("."), Path(".."), Path("/kaggle/working"), Path("/kaggle/working/kaggle_notebooks")]:
    if p.exists() and str(p.resolve()) not in sys.path:
        sys.path.insert(0, str(p.resolve()))

try:
    from src.config import (
        ALL_RAW_FILES, REPORTS_DIR, FIGURES_DIR, TABLES_DIR, DATA_PROCESSED,
        GAME_TRACKING_FILES, print_file_status, ensure_dirs, IN_KAGGLE, DATA_RAW
    )
    from src.data_loading import (
        load_players, load_games, load_career_successes,
        load_combine_results, load_combine_tracking, load_player_play,
        quick_schema
    )
    from src.player_mapping import validate_player_joins, print_join_report, validate_combine_tracking
    from src.visualization import plot_missing_data_heatmap
except ImportError:
    # Inline fallback if src package is not mounted alongside the notebook
    IN_KAGGLE = Path("/kaggle").exists()
    PROJECT_ROOT = Path("/kaggle/working") if IN_KAGGLE else Path(".").resolve()
    REPORTS_DIR = PROJECT_ROOT / "outputs" / "reports"
    FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
    TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
    DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
    
    # Locate raw data
    raw_candidates = [
        Path("/kaggle/input/nfl-big-data-bowl-2027"),
        Path("/kaggle/input/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027"),
        Path("./data/raw/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027"),
        Path("./data/raw/nfl-big-data-bowl-2027"),
        Path("./data/raw"),
        Path("../data/raw"),
    ]
    DATA_RAW = raw_candidates[0]
    for c in raw_candidates:
        if (c / "players.csv").exists():
            DATA_RAW = c
            break
    if not (DATA_RAW / "players.csv").exists() and Path("/kaggle/input").exists():
        found = list(Path("/kaggle/input").glob("**/players.csv"))
        if found:
            DATA_RAW = found[0].parent
            
    ALL_RAW_FILES = {
        "players": DATA_RAW / "players.csv",
        "combine_results": DATA_RAW / "combine_results.csv",
        "combine_tracking": DATA_RAW / "combine_tracking.csv",
        "player_career_successes": DATA_RAW / "player_career_successes.csv",
        "player_play": DATA_RAW / "player_play.csv",
        "games": DATA_RAW / "games.csv",
        "game_tracking_2023": DATA_RAW / "game_tracking_2023.csv",
        "game_tracking_2024": DATA_RAW / "game_tracking_2024.csv",
        "game_tracking_2025": DATA_RAW / "game_tracking_2025.csv",
    }
    GAME_TRACKING_FILES = {
        2023: DATA_RAW / "game_tracking_2023.csv",
        2024: DATA_RAW / "game_tracking_2024.csv",
        2025: DATA_RAW / "game_tracking_2025.csv",
    }
    
    def ensure_dirs():
        for d in [REPORTS_DIR, FIGURES_DIR, TABLES_DIR, DATA_PROCESSED]:
            d.mkdir(parents=True, exist_ok=True)
            
    def print_file_status():
        print(f"{'File':<30} {'Status':<18} {'Size'}")
        print("-" * 55)
        for name, path in ALL_RAW_FILES.items():
            if path.exists():
                size_mb = path.stat().st_size / (1024 * 1024)
                print(f"{name:<30} ✓ FOUND {size_mb:>15.1f} MB")
            else:
                print(f"{name:<30} ✗ MISSING")

    def quick_schema(path, n_rows=3):
        df = pd.read_csv(path, nrows=n_rows)
        dtypes = pd.DataFrame({'column': df.columns, 'dtype': df.dtypes.astype(str).values, 'sample_value': df.iloc[0].values})
        return df, dtypes
        
    def load_players():
        return pd.read_csv(ALL_RAW_FILES['players'])
    def load_games():
        return pd.read_csv(ALL_RAW_FILES['games'])
    def load_career_successes():
        return pd.read_csv(ALL_RAW_FILES['player_career_successes'])
    def load_combine_results(cols=None):
        return pd.read_csv(ALL_RAW_FILES['combine_results'], usecols=cols)
    def load_combine_tracking():
        return pd.read_csv(ALL_RAW_FILES['combine_tracking'])
    def load_player_play():
        return pd.read_csv(ALL_RAW_FILES['player_play'])
        
    def validate_combine_tracking(df):
        return {'n_rows': len(df), 'n_players': df['nfl_id'].nunique(), 'n_drill_types': df['drill_type'].nunique()}
        
    def validate_player_joins(players, combine_results=None, combine_tracking=None, player_play=None, career_successes=None):
        m = set(players['nfl_id'].unique())
        res = {'n_players_master': len(m)}
        for name, df in [('combine_results', combine_results), ('combine_tracking', combine_tracking), ('player_play', player_play), ('career_successes', career_successes)]:
            if df is not None and 'nfl_id' in df.columns:
                s = set(df['nfl_id'].dropna().unique())
                res[f'{name}_n'] = len(s)
                res[f'{name}_in_master'] = len(s & m)
        return res
        
    def print_join_report(rep):
        print('=== Player Join Validation Report ===')
        for k, v in rep.items():
            print(f'  {k}: {v}')
            
    def plot_missing_data_heatmap(df, title='', filename=''):
        plt.figure(figsize=(10, 4))
        sns.heatmap(df.isnull().iloc[:1000], cbar=False, cmap='viridis')
        plt.title(title)
        if filename:
            out_p = FIGURES_DIR / filename
            out_p.parent.mkdir(parents=True, exist_ok=True)
            plt.savefig(out_p, bbox_inches='tight', dpi=100)
        plt.show()

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(name)s | %(message)s')
ensure_dirs()

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Local'}")
print(f"Raw data directory: {DATA_RAW}")
print(f"Reports directory:  {REPORTS_DIR}")
print()
print_file_status()

## 1. Quick Schema Peek

Before loading full files into memory, inspect column names and sample values from each dataset.

In [ ]:
schemas = {}

for name, path in ALL_RAW_FILES.items():
    if not path.exists():
        print(f'✗ {name}: FILE NOT FOUND — {path}')
        continue
    head, dtypes = quick_schema(path, n_rows=3)
    schemas[name] = dtypes
    print('\n' + '=' * 60)
    print(f'  {name}  ({path.stat().st_size / 1_048_576:.1f} MB)')
    print(f'  Columns ({len(head.columns)}):')
    display(dtypes)
    print('  Head:')
    display(head)

## 2. Load Core Metadata & Combine Results

In [ ]:
players = None
games = None
career_successes = None
combine_results = None

try:
    players = load_players()
    print(f'players: {len(players):,} rows × {players.shape[1]} cols')
    display(players.head(3))
except Exception as e:
    print(f'players load error: {e}')

try:
    games = load_games()
    print(f'\ngames: {len(games):,} rows × {games.shape[1]} cols')
    display(games.head(3))
except Exception as e:
    print(f'games load error: {e}')

try:
    career_successes = load_career_successes()
    print(f'\ncareer_successes: {len(career_successes):,} rows × {career_successes.shape[1]} cols')
    display(career_successes.head(3))
except Exception as e:
    print(f'career_successes load error: {e}')

try:
    combine_results = load_combine_results(cols=None)
    print(f'\ncombine_results: {len(combine_results):,} rows × {combine_results.shape[1]} cols')
    display(combine_results.head(3))
except Exception as e:
    print(f'combine_results load error: {e}')

## 3. Players — Detailed Profile

In [ ]:
if players is not None:
    print('=== PLAYERS TABLE ===')
    print(f'Shape: {players.shape}')
    print(f'Unique nfl_id: {players["nfl_id"].nunique()}')
    print(f'Null nfl_id: {players["nfl_id"].isna().sum()}')
    print(f'nfl_id unique? {players["nfl_id"].is_unique}')
    print()

    miss = players.isnull().sum()
    miss = miss[miss > 0]
    if not miss.empty:
        print('Columns with missing values (%):')
        display((miss / len(players) * 100).to_frame('missing_pct'))
    else:
        print('No missing values in players.')

    pos_col = 'position' if 'position' in players.columns else ('nfl_position' if 'nfl_position' in players.columns else None)
    if pos_col:
        print(f'\nPosition distribution ({pos_col}):')
        display(players[pos_col].value_counts().to_frame())

    print('\nNumeric summary:')
    display(players.describe())

## 4. Combine Results — Detailed Profile

In [ ]:
if combine_results is not None:
    print('=== COMBINE RESULTS ===')
    print(f'Shape: {combine_results.shape}')
    print(f'Columns: {combine_results.columns.tolist()}')
    print()

    miss_pct = (combine_results.isnull().sum() / len(combine_results) * 100).sort_values(ascending=False)
    miss_pct = miss_pct[miss_pct > 0]
    print('Missing values (%):')
    display(miss_pct.to_frame('missing_pct'))

    print('\nNumeric summary:')
    display(combine_results.describe())

    plot_missing_data_heatmap(combine_results, title='Missing Data — combine_results', filename='missing_combine_results.png')

## 5. Combine Tracking — Schema & Validation

In [ ]:
combine_tracking = None

try:
    combine_tracking = load_combine_tracking()
    print(f'combine_tracking: {len(combine_tracking):,} rows × {combine_tracking.shape[1]} cols')
    print(f'Memory: {combine_tracking.memory_usage(deep=True).sum() / 1_048_576:.1f} MB')
    print()
    display(combine_tracking.head())

    val = validate_combine_tracking(combine_tracking)
    print('\nCombine Tracking Validation:')
    for k, v in val.items():
        if k != 'complete_player_ids':
            print(f'  {k}: {v}')
except Exception as e:
    print(f'combine_tracking error: {e}')

In [ ]:
if combine_tracking is not None:
    print('=== DRILL TYPE DISTRIBUTION ===')
    if 'drill_type' in combine_tracking.columns:
        drill_counts = combine_tracking.groupby('drill_type').agg(
            n_rows=('nfl_id', 'count'),
            n_players=('nfl_id', 'nunique'),
        ).sort_values('n_players', ascending=False)
        display(drill_counts)

    if 'drill_name' in combine_tracking.columns:
        print('\nTop drill names:')
        display(combine_tracking['drill_name'].value_counts().head(15).to_frame())

    plot_missing_data_heatmap(combine_tracking.iloc[:20000], title='Missing Data — combine_tracking sample', filename='missing_combine_tracking.png')

## 6. Player Play — Schema & Basic Stats

In [ ]:
player_play = None

try:
    player_play = load_player_play()
    print(f'player_play: {len(player_play):,} rows × {player_play.shape[1]} cols')
    print(f'Memory: {player_play.memory_usage(deep=True).sum() / 1_048_576:.1f} MB')
    print(f'Unique players: {player_play["nfl_id"].nunique()}')
    if 'game_id' in player_play.columns:
        print(f'Unique games: {player_play["game_id"].nunique()}')
    display(player_play.head(3))

    miss_pct = (player_play.isnull().sum() / len(player_play) * 100).sort_values(ascending=False)
    miss_pct = miss_pct[miss_pct > 0]
    print('\nMissing values (%):')
    display(miss_pct.to_frame('missing_pct'))
except Exception as e:
    print(f'player_play error: {e}')

## 7. Game Tracking — Schema Inspection

In [ ]:
print('=== GAME TRACKING FILES ===')
for season, path in GAME_TRACKING_FILES.items():
    if path.exists():
        size_mb = path.stat().st_size / 1_048_576
        head, dtypes = quick_schema(path, n_rows=3)
        print(f'\ngame_tracking_{season}: {size_mb:.0f} MB | {len(head.columns)} columns')
        display(dtypes)
    else:
        print(f'game_tracking_{season}: FILE NOT FOUND — {path}')

## 8. Player Join Validation

In [ ]:
join_report_data = {}
if players is not None:
    join_report_data = validate_player_joins(
        players=players,
        combine_results=combine_results,
        combine_tracking=combine_tracking,
        player_play=player_play,
        career_successes=career_successes,
    )
    print_join_report(join_report_data)

    if combine_tracking is not None and player_play is not None:
        ct_ids = set(combine_tracking['nfl_id'].dropna().unique())
        pp_ids = set(player_play['nfl_id'].dropna().unique())
        both = ct_ids & pp_ids
        print(f'\nPlayers with BOTH combine tracking AND NFL player_play records: {len(both)}')
        print('This forms the foundational scouting cohort for BDB 2027.')
else:
    print('Cannot run join validation — players table not loaded.')

## 9. Generate & Export data_inventory.md Report

In [ ]:
def generate_inventory_report(
    players, combine_results, combine_tracking, player_play,
    career_successes, games, join_report,
) -> str:
    from datetime import datetime
    lines = []
    lines.append('# NFL Big Data Bowl 2027 — Data Inventory Report')
    lines.append(f'\n*Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}*\n')
    lines.append('## File Sizes')
    lines.append('| File | Status | Size |')
    lines.append('|------|--------|------|')
    for name, path in ALL_RAW_FILES.items():
        if path.exists():
            size_mb = path.stat().st_size / 1_048_576
            lines.append(f'| {name} | ✓ Found | {size_mb:.1f} MB |')
        else:
            lines.append(f'| {name} | ✗ MISSING | — |')

    def df_summary(name, df):
        if df is None:
            return [f'\n## {name}\n\n**NOT LOADED**\n']
        rows = [f'\n## {name}\n']
        rows.append(f'- **Rows**: {len(df):,}')
        rows.append(f'- **Columns**: {df.shape[1]}')
        mem_mb = df.memory_usage(deep=True).sum() / 1_048_576
        rows.append(f'- **Memory**: {mem_mb:.1f} MB')
        if 'nfl_id' in df.columns:
            rows.append(f'- **Unique players**: {df["nfl_id"].nunique():,}')
        return rows

    for name, df in [
        ('players', players), ('combine_results', combine_results),
        ('combine_tracking', combine_tracking), ('player_play', player_play),
        ('career_successes', career_successes), ('games', games),
    ]:
        lines.extend(df_summary(name, df))

    if join_report:
        lines.append('\n## Join Validation')
        lines.append(f'- Master player table: **{join_report.get("n_players_master", "N/A")}** players')
        for prefix in ['combine_results', 'combine_tracking', 'player_play', 'career_successes']:
            n_key = f'{prefix}_n'
            if n_key in join_report:
                lines.append(f'- {prefix}: **{join_report[n_key]}** players ({join_report.get(f"{prefix}_in_master", "?")} in master)')

    return '\n'.join(lines)

report_text = generate_inventory_report(
    players, combine_results, combine_tracking, player_play,
    career_successes, games, join_report_data,
)

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
report_path = REPORTS_DIR / 'data_inventory.md'
report_path.write_text(report_text, encoding='utf-8')
print(f'Report successfully saved to: {report_path}')
print()
print(report_text[:1200], '...(truncated for display)')